# Etapa 2 — Pré-processamento e Extração de Features

**Por que estas duas etapas juntas?**

O dataset completo (~10 GB) não cabe na memória de uma vez. A solução é processar **uma classe por vez**:
1. Carregar instâncias da Classe 0 → limpar → extrair features → salvar → liberar memória
2. ... repetir para as 10 classes

**Saídas geradas por este notebook:**

| Arquivo | Conteúdo | Usado em |
|---------|----------|----------|
| `cleaned.parquet` | Séries temporais limpas (forward-fill, z-score) | Base para extração de features |
| `features.parquet` | 88 features × janela, rótulo = tipo de falha (10 classes) | Referência / Abordagem 1 |
| `features_window_class.parquet` | 88 features × janela, rótulo = estado operacional (17 classes) | **Input dos modelos (Etapas 4–6)** |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold

from config import (
    CLEANED_DATA_PATH,
    FAULT_CLASSES,
    FEATURES_DATA_PATH,
    FFILL_LIMIT,
    N_INSTANCES_VALIDATION,
    N_SPLITS_CV,
    VALIDATION_MODE,
)
from src.data_loader import load_sample
from src.feature_engineering import run_pipeline_chunked

# Banner de modo — mudança rápida em config.py
mode_label = 'VALIDACAO' if VALIDATION_MODE else 'COMPLETO'
n_inst = N_INSTANCES_VALIDATION if VALIDATION_MODE else 'todas'
print(f"{'='*55}")
print(f"  Modo: {mode_label}  |  Instancias por classe: {n_inst}")
if VALIDATION_MODE:
    print("  Para rodar o dataset completo:")
    print("  config.py -> VALIDATION_MODE = False")
print(f"{'='*55}")

## 2.1 Conceito: forward-fill (preenchimento de lacunas)

Antes de rodar o pipeline, vamos entender o que o forward-fill faz em um exemplo real.

Imagine que o sensor de pressão 'parou' por 30 segundos e voltou. Durante esse tempo,
a leitura fica como NaN (desconhecida). O forward-fill copia o último valor válido para
preencher esses buracos — mas só por até 60 segundos. Buracos maiores permanecem como NaN.

In [ ]:
# Carregar uma instância de exemplo para demonstrar
df_demo = load_sample(n_instances_per_class=1)
df_inst = df_demo[df_demo['instance_id'] == df_demo['instance_id'].iloc[0]].copy()

sensor = 'P-TPT'
if sensor in df_inst.columns:
    before = df_inst[sensor].isna().sum()
    df_inst[sensor] = df_inst[sensor].ffill(limit=FFILL_LIMIT)
    after = df_inst[sensor].isna().sum()
    print(f'Sensor {sensor}:')
    print(f'  NaN antes do forward-fill : {before}')
    print(f'  NaN depois do forward-fill: {after}')
    print(f'  Lacunas preenchidas       : {before - after}')
else:
    print(f'Sensor {sensor} nao encontrado nessa instancia.')

## 2.2 Conceito: GroupKFold (divisão treino/teste por poço)

Este é um ponto crucial para a validade do TCC.

Se dividirmos os dados aleatoriamente, janelas do **mesmo poço** podem aparecer no treino
e no teste ao mesmo tempo. O modelo 'memoriza' aquele poço e parece ótimo — mas falha em
poços novos que nunca viu. Isso é chamado de **vazamento de dados** (*data leakage*).

O GroupKFold garante que cada poço aparece em **apenas um** dos conjuntos.

In [ ]:
# Demonstrar GroupKFold com os poucos dados da amostra
instances = df_demo['instance_id'].unique()
groups = df_demo.groupby('instance_id')['fault_class'].first().loc[instances].values

# A demo usa no máximo 3 folds para funcionar com qualquer tamanho de amostra.
# No treino real usamos N_SPLITS_CV do config (5 no modo completo).
n_splits_demo = min(3, len(instances))
gkf_demo = GroupKFold(n_splits=n_splits_demo)

print(f'GroupKFold com {n_splits_demo} folds — cada poco aparece em apenas 1 fold de teste:')
for fold, (train_idx, test_idx) in enumerate(gkf_demo.split(instances, groups, groups)):
    print(f'  Fold {fold+1}: treino={len(train_idx)} instancias | teste={len(test_idx)} instancias')

print(f'\n(No treino real usamos N_SPLITS_CV={N_SPLITS_CV} folds com todas as instancias)')

## 2.3 Executar o pipeline em partes

Agora rodamos o pipeline completo: limpeza + extração de features, classe por classe.

No **modo validação**, são carregadas apenas `N_INSTANCES_VALIDATION` instâncias por classe
(definido em `config.py`). Isso é suficiente para verificar se o código funciona sem erros,
em segundos ou poucos minutos.

No **modo completo** (`VALIDATION_MODE = False`), todas as instâncias são processadas.
Isso pode levar 30 minutos a algumas horas dependendo do hardware.

In [ ]:
# Apagar arquivos anteriores se existirem (evita acumular dados de runs diferentes)
for path in [CLEANED_DATA_PATH, FEATURES_DATA_PATH]:
    if path.exists():
        path.unlink()
        print(f'Arquivo anterior removido: {path.name}')

print('\nIniciando pipeline...\n')
run_pipeline_chunked(verbose=True)
print('\nPipeline concluido!')

## 2.4 Verificar os arquivos gerados

In [ ]:
# Carregar e inspecionar o arquivo de dados limpos
df_clean = pd.read_parquet(CLEANED_DATA_PATH)
print('=== Dados Limpos ===')
print(f'Shape          : {df_clean.shape}')
print(f'Instancias     : {df_clean["instance_id"].nunique()}')
print()

dist = df_clean.groupby(['fault_class', 'source_type'])['instance_id'].nunique().unstack(fill_value=0)
dist.index = dist.index.map(lambda c: f'{c} — {FAULT_CLASSES[c]}')
print('Instancias por classe e fonte:')
print(dist.to_string())

In [ ]:
# Carregar e inspecionar o arquivo de features
df_features = pd.read_parquet(FEATURES_DATA_PATH)
META_COLS = ['instance_id', 'fault_class', 'source_type', 'window_start']
n_features = df_features.shape[1] - len(META_COLS)

print('=== Features ===')
print(f'Shape          : {df_features.shape}')
print(f'Janelas totais : {len(df_features):,}')
print(f'Features/janela: {n_features}')
print()
print('Janelas por classe:')
print(df_features['fault_class'].value_counts().sort_index()
      .rename(FAULT_CLASSES).to_string())

## 2.5 Rotulagem por Estado Operacional — features_window_class.parquet

O `features.parquet` usa **rotulagem por instância**: toda janela de um poço com DHSV recebe `fault_class=2`, mesmo as janelas de operação normal antes da falha.

Para classificação em tempo real precisamos da **rotulagem por estado operacional**: cada janela recebe a moda da coluna `class` dentro dela — distinguindo normal (0), transiente (101–109) e ativo (1–9).

A função `run_pipeline_from_cleaned()` lê o `cleaned.parquet` já gerado e produz `features_window_class.parquet` sem re-processar os dados brutos.

In [ ]:
from config import FEATURES_WINDOW_PATH, WINDOW_CLASSES
from src.feature_engineering import run_pipeline_from_cleaned

# Remover arquivo anterior se existir
if FEATURES_WINDOW_PATH.exists():
    FEATURES_WINDOW_PATH.unlink()
    print(f'Arquivo anterior removido: {FEATURES_WINDOW_PATH.name}')

print('Gerando features com rotulagem por estado operacional (17 classes)...\n')
run_pipeline_from_cleaned(
    cleaned_path=CLEANED_DATA_PATH,
    features_path=FEATURES_WINDOW_PATH,
    label_strategy='window',
    verbose=True,
)

# Verificar saída
df_wc = pd.read_parquet(FEATURES_WINDOW_PATH)
META_WC = ['instance_id', 'fault_class', 'window_label', 'source_type', 'window_start']
n_feat = df_wc.shape[1] - len(META_WC)

print(f'\n=== features_window_class.parquet ===')
print(f'Shape          : {df_wc.shape}')
print(f'Janelas totais : {len(df_wc):,}')
print(f'Features/janela: {n_feat}')
print(f'\nJanelas por estado operacional:')
for cls, count in df_wc['window_label'].value_counts().sort_index().items():
    label = WINDOW_CLASSES.get(cls, str(cls))
    pct = 100 * count / len(df_wc)
    print(f'  {cls:>3} — {label:<30}: {count:>6,} ({pct:.1f}%)')